In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA

###############################################################################
# Paths & constants
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")

OUT_TRIALS  = Path("trial_level_cca_fixedlag.csv")
OUT_SUBJECT = Path("subject_best_lag.csv")

SUBJECTS = np.setdiff1d(np.arange(32, 98), [32, 37, 53, 61, 66, 78, 84, 94, 96])
SHIFTS = np.arange(-100, 101)      # ±1 s at 100 Hz → samples
WIN_OFFSET1 = 200                  # discard first 200 ms
WIN_OFFSET2 = 110                  # discard last 110 ms

###############################################################################
# Helper functions
###############################################################################

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over time×channels."""
    x = x - x.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.mean(x**2))
    return x / scale


def cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> float:
    """Canonical correlation (single component)."""
    cca = CCA(n_components=1, max_iter=1000)
    cca.fit(eeg, pupil)
    u, v = cca.transform(eeg, pupil)
    return float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])


In [4]:
def load_all_trials(
        sub: int,
        eeg_root: Path = EEG_ROOT,
        pupil_root: Path = PUPIL_ROOT,
        min_len: int = 40,
        max_len_diff: int = 30,
) -> list[tuple[np.ndarray, np.ndarray, dict]]:
    """
    Load *all* valid EEG-pupil trial pairs for one subject.

    Parameters
    ----------
    sub : int
        Numeric subject ID (e.g. 42).
    eeg_root, pupil_root : Path
        Roots of the pre-processed EEG and pupil folders.
    min_len : int
        Minimum number of samples a pupil trace must have to be accepted.
    max_len_diff : int
        Reject trial if |len(pupil)-len(eeg)| exceeds this.
    Returns
    -------
    trials : list of (eeg, pupil_z, meta)
        * eeg        – (T × n_channels) float64, already centred/scaled
        * pupil_z    – (T × 1) float64, per-trial z-scored
        * meta       – dict with subject/condition/load/epoch
    """
    trials = []
    sub_tag = f"sub-{sub:03d}"
    eeg_sub  = eeg_root   / sub_tag
    pupil_sub = pupil_root / sub_tag

    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        return trials

    # iterate condition (“control” / “memory”) and load (“05” / “09” / “13”)
    for cond_path in sorted(eeg_sub.iterdir()):
        if not cond_path.is_dir():
            continue
        for load_path in sorted(cond_path.iterdir()):
            if not load_path.is_dir():
                continue

            # matching pupil directory
            pupil_path = pupil_sub / cond_path.name / load_path.name
            if not pupil_path.exists():
                continue

            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))
            common = {f.name for f in eeg_epochs} & {f.name for f in pupil_epochs}
            if not common:
                continue

            for fname in sorted(common):
                eeg_df   = pd.read_csv(load_path / fname,  comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#",
                                       names=["time", "diameter_z"], index_col=0)

                eeg   = eeg_df.values.astype(float)
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic validity checks
                if len(pupil) < min_len or abs(len(pupil) - len(eeg)) > max_len_diff:
                    continue

                # normalise signals ----------------------------------------
                eeg_norm = normalise_eeg(eeg)           # your helper from before
                pupil_z  = ((pupil - pupil.mean()) / pupil.std(ddof=0))

                # centre window to protect against wrap-around after shifts
                T = min(len(eeg_norm), len(pupil_z))
                eeg_norm = eeg_norm[0:T, :]  # (T × n_channels)
                pupil_z  = pupil_z[0:T].reshape(-1, 1)

                meta = {
                    "subject":   sub_tag,
                    "condition": cond_path.name,
                    "load":      int(load_path.name),
                    "epoch":     fname
                }
                trials.append((eeg_norm, pupil_z, meta))

    return trials


In [5]:
trials = load_all_trials(33)

In [6]:
eeg_trial = trials[0][0]
pupil_trial = trials[0][1]

In [ ]:
def search_best_lag(
        train: list[tuple[np.ndarray, np.ndarray, dict]],
        lam: float,
        shifts: np.ndarray = SHIFTS) -> tuple[float, int]:
    """
    Search for the best lag shift in a training set.

    Parameters
    ----------
    train : list of (eeg, pupil_z, meta)
        Training set with EEG and pupil traces.
    shifts : np.ndarray
        Array of lag shifts to test.
    Returns
    -------
    best_corr : float
        Best canonical correlation found.
    best_shift : int
        Lag shift that maximised the correlation.
    """
    per_shift_r = {}
    for shift in shifts:
        r = []
        for eeg, pupil_z, _ in train:
            if shift < 0:
                eeg_shifted = eeg[:shift]
                pupil_shifted = pupil_z[-shift:]
            elif shift > 0:
                eeg_shifted = eeg[shift:]
                pupil_shifted = pupil_z[:-shift]
            else:
                eeg_shifted = eeg
                pupil_shifted = pupil_z*lam

            r.append(cca_corr(eeg_shifted, pupil_shifted))
        per_shift_r[shift] = np.mean(r)

    best_shift = max(per_shift_r, key=per_shift_r.get)
    return per_shift_r[best_shift], best_shift

In [ ]:
for subj in SUBJECTS:
    trials = load_all_trials(subj)                         # list of (eeg, pupil)

    # ----- split trials ↯ -----------------------------------------------
    train_idx = trials[::2]         # even trials
    test_idx  = trials[1::2]        # odd  trials

    # ----- lag search on TRAIN ------------------------------------------
    best_shift = search_best_lag(train_idx, SHIFTS, lam=0.1)

    # ----- fit weights ONCE using all train trials at best_shift ---------
    X_train, Y_train = concat_trials(train_idx, shift=best_shift)
    cca = rCCA(latent_dims=1, c=[0.1, 0.1]).fit(X_train, Y_train)

    # ----- per-trial r on TEST with frozen lag & weights -----------------
    for eeg, pupil, meta in iterate_trials(test_idx):
        u, v = cca.transform(roll(eeg, best_shift)[win], pupil_z[win])
        meta['r'] = corr(u[:, 0], v[:, 0])
        store(meta)
    # (optional) swap train/test and repeat so every trial gets a score


In [ ]:
###############################################################################
# Main
###############################################################################

trial_rows   = []
subject_rows = []

for sub in SUBJECTS:
    sub_tag = f"sub-{sub:03d}"
    eeg_sub = EEG_ROOT / sub_tag
    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing – skipped")
        continue

    # ---------------------------------------------------------------------
    # Pass 1 – compute CCA correlations for every shift & every trial
    # ---------------------------------------------------------------------
    shift_to_corrs: dict[int, list[float]] = {s: [] for s in SHIFTS}
    trial_cache   = []  # will store (meta_dict, {shift: r})

    for cond_path in eeg_sub.iterdir():          # memory / control
        for load_path in cond_path.iterdir():    # 05 / 09 / 13
            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_path   = (PUPIL_ROOT / sub_tag / cond_path.name / load_path.name)
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))

            common = {e.name for e in eeg_epochs} & {p.name for p in pupil_epochs}
            if not common:
                print(f"{sub_tag} {cond_path.name} {load_path.name}: no common epochs - skipped")
                continue
            print(f"{sub_tag} {cond_path.name} {load_path.name}: {len(common)} common epochs")

            for fname in common:
                eeg_df   = pd.read_csv(load_path / fname, comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#", names=["time", "diameter_z"], index_col=0)

                eeg   = normalise_eeg(eeg_df.values.astype(float))
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic sanity
                if len(pupil) < 40 or abs(len(pupil) - len(eeg)) > 30:
                    print(f"{sub_tag} {cond_path.name} {load_path.name} {fname}: invalid trial - skipped")
                    continue

                # common window
                T   = min(len(eeg), len(pupil))
                win = slice(WIN_OFFSET1, T - WIN_OFFSET2)
                pupil_z = ((pupil - pupil.mean()) / pupil.std(ddof=0))[win].reshape(-1, 1)

                per_shift_r = {}
                for s in SHIFTS:
                    eeg_shift = np.roll(eeg, s, axis=0)[win]
                    r = cca_corr(eeg_shift, pupil_z)
                    per_shift_r[s] = r
                    shift_to_corrs[s].append(r)

                trial_cache.append((
                    {
                        "subject": sub_tag,
                        "condition": cond_path.name,
                        "load": int(load_path.name),
                        "epoch": fname
                    },
                    per_shift_r
                ))

    if not trial_cache:  # subject had no usable trials
        continue

    # ---------------------------------------------------------------------
    # Decide on the subject's best lag (max mean r over trials)
    # ---------------------------------------------------------------------
    mean_r_per_shift = {s: np.mean(vals) for s, vals in shift_to_corrs.items()}
    best_shift = max(mean_r_per_shift, key=mean_r_per_shift.get)
    best_lag_ms = best_shift * 10

    subject_rows.append({
        "subject": sub_tag,
        "lag_ms": best_lag_ms,
        "n_trials": sum(len(v) for v in shift_to_corrs.values()) // len(SHIFTS),
        "mean_r": mean_r_per_shift[best_shift]
    })

    # ---------------------------------------------------------------------
    # Pass 2 – store trial‑level r using that fixed lag
    # ---------------------------------------------------------------------
    for meta, per_shift in trial_cache:
        meta["r"] = per_shift[best_shift]
        meta["lag_ms"] = best_lag_ms
        trial_rows.append(meta)

    print(f"{sub_tag}: best lag {best_lag_ms:+d} ms (mean r={mean_r_per_shift[best_shift]:.3f})")

# -------------------------------------------------------------------------
print(f"Finished – {len(trial_rows)} trials from {len(subject_rows)} subjects")

pd.DataFrame(trial_rows).to_csv(OUT_TRIALS, index=False)
pd.DataFrame(subject_rows).to_csv(OUT_SUBJECT, index=False)
print(f"Saved {OUT_TRIALS} and {OUT_SUBJECT}")
